In [ ]:
import pandas as pd
import os
from functools import reduce

data_dir = "data"
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

csv_files = sorted([f for f in os.listdir(data_dir) if f.endswith(".csv")])

all_long = []

for file in csv_files:
    path = os.path.join(data_dir, file)
    df = pd.read_csv(path)
    country_col = df.columns[0]
    df_long = df.reset_index(drop=True).melt(id_vars=country_col, var_name="year", value_name=os.path.splitext(file)[0])
    tmp = df_long["year"].astype(str).str.replace("YR", "", regex=False)
    numeric = pd.to_numeric(tmp, errors="coerce")
    if numeric.notnull().all():
        df_long["year"] = numeric.astype(int)
    else:
        df_long["year"] = tmp
    all_long.append((df_long, country_col))

if not all_long:
    raise SystemExit("no csv files found")

base_df, base_country = all_long[0]

for right_df, right_country in all_long[1:]:
    merged = pd.merge(base_df, right_df, left_on=[base_country, "year"], right_on=[right_country, "year"], how="outer")
    if right_country != base_country and right_country in merged.columns:
        merged.drop(columns=[right_country], inplace=True)
    base_df = merged

base_df.to_csv(os.path.join(output_dir, "merged_worldbank_long.csv"), index=False)
print(base_df.columns)


  economy  year  worldbank_age_0_14_share  worldbank_age_65_plus_share  \
0     ABW  1960                 42.512108                     2.855868   
1     ABW  1961                 42.175482                     2.870714   

   worldbank_crude_birth_rate  worldbank_crude_death_rate  \
0                      32.043                       7.525   
1                      31.225                       7.347   

   worldbank_fertility_rate  worldbank_gdp  worldbank_gdp_per_capita  \
0                     4.567            NaN                       NaN   
1                     4.422            NaN                       NaN   

   worldbank_inflation  worldbank_life_expectancy  worldbank_net_migration  \
0                  NaN                     64.049                   -788.0   
1                  NaN                     64.215                   -566.0   

   worldbank_population  worldbank_school_tertiary  \
0               54922.0                        NaN   
1               55578.0          

In [13]:
maddison_path = os.path.join(data_dir, "maddison2023_web.dta")
maddison_df = pd.read_stata(maddison_path, convert_categoricals=False)
print(maddison_df.columns)

Index(['countrycode', 'country', 'region', 'year', 'gdppc', 'pop'], dtype='object')


In [ ]:
import pandas as pd
import os

data_dir = "data"
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

merged_csv_path = os.path.join(output_dir, "merged_worldbank_long.csv")
merged_df = pd.read_csv(merged_csv_path, dtype=str)

maddison_path = os.path.join(data_dir, "maddison2023_web.dta")
maddison_df = pd.read_stata(maddison_path, convert_categoricals=False)

base_country_col = merged_df.columns[0]
if "year" not in merged_df.columns:
    raise SystemExit("merged file has no 'year' column")

merged_df["year"] = pd.to_numeric(merged_df["year"], errors="coerce")
maddison_df["year"] = pd.to_numeric(maddison_df["year"], errors="coerce")

if "countrycode" in maddison_df.columns:
    right_country = "countrycode"
elif "country" in maddison_df.columns:
    right_country = "country"
else:
    raise SystemExit("maddison file has no country column")

desired_gdppc = "gdppc_maddison" if "gdppc" in merged_df.columns else "gdppc"
right = maddison_df[[right_country, "year", "gdppc"]].rename(columns={"gdppc": desired_gdppc})

merged_df["_m_base"] = merged_df[base_country_col].astype(str).str.strip().str.upper()
right["_m_right"] = right[right_country].astype(str).str.strip().str.upper()

result = pd.merge(
    merged_df,
    right[["_m_right", "year", desired_gdppc]],
    left_on=["_m_base", "year"],
    right_on=["_m_right", "year"],
    how="left",
)

if "_m_base" in result.columns:
    result.drop(columns=["_m_base"], inplace=True)
if "_m_right" in result.columns:
    result.drop(columns=["_m_right"], inplace=True)

result.to_csv(os.path.join(output_dir, "merged_worldbank_long_with_gdppc.csv"), index=False)
print("saved:", os.path.join(output_dir, "merged_worldbank_long_with_gdppc.csv"))
print(result.columns)

saved: output/merged_worldbank_long_with_gdppc.csv
Index(['economy', 'year', 'worldbank_age_0_14_share',
       'worldbank_age_65_plus_share', 'worldbank_crude_birth_rate',
       'worldbank_crude_death_rate', 'worldbank_fertility_rate',
       'worldbank_gdp', 'worldbank_gdp_per_capita', 'worldbank_inflation',
       'worldbank_life_expectancy', 'worldbank_net_migration',
       'worldbank_population', 'worldbank_school_tertiary',
       'worldbank_under5_mortality', 'worldbank_unemployment',
       'worldbank_urban_share', 'gdppc'],
      dtype='object')
  economy  year worldbank_age_0_14_share worldbank_age_65_plus_share  \
0     ABW  1960         42.5121080805506            2.85586832234806   
1     ABW  1961         42.1754815261297            2.87071439495488   

  worldbank_crude_birth_rate worldbank_crude_death_rate  \
0                     32.043                      7.525   
1                     31.225                      7.347   

  worldbank_fertility_rate worldbank_gdp w

In [16]:
import pandas as pd
import os

output_dir = "output"
merged_path = os.path.join(output_dir, "merged_worldbank_long_with_gdppc.csv")
df = pd.read_csv(merged_path)

drop_cols = [
    "worldbank_gdp_per_capita",
    "worldbank_unemployment",
    "worldbank_inflation",
    "worldbank_school_tertiary",
    "worldbank_under5_mortality",
    "worldbank_gdp"
]

df = df.drop(columns=[c for c in drop_cols if c in df.columns])

df = df.rename(columns=lambda x: x.replace("worldbank_", "") if x.startswith("worldbank_") else x)

cleaned_path = os.path.join(output_dir, "merged_worldbank_long_cleaned.csv")
df.to_csv(cleaned_path, index=False)
print("saved:", cleaned_path)
print(df.columns)

saved: output/merged_worldbank_long_cleaned.csv
Index(['economy', 'year', 'age_0_14_share', 'age_65_plus_share',
       'crude_birth_rate', 'crude_death_rate', 'fertility_rate',
       'life_expectancy', 'net_migration', 'population', 'urban_share',
       'gdppc'],
      dtype='object')


In [25]:
import pandas as pd
import os

output_dir = "output"
cleaned_path = os.path.join(output_dir, "merged_worldbank_long_cleaned.csv")
df = pd.read_csv(cleaned_path)

df['year'] = pd.to_numeric(df['year'], errors='coerce')
mask = (df['year'] >= 1960) & (df['year'] <= 2022)
subset = df[mask]

value_cols = subset.columns.difference([df.columns[0], 'year'])

nan_countries = subset.groupby(df.columns[0])[value_cols].apply(lambda x: x.isna().any(axis=1).any())

count_nan_countries = nan_countries.sum()

print("Количество стран с хотя бы одним NaN за 1960-2022:", count_nan_countries)


Количество стран с хотя бы одним NaN за 1960-2022: 122


In [27]:
import pandas as pd
import os

output_dir = "output"
cleaned_path = os.path.join(output_dir, "merged_worldbank_long_cleaned.csv")
df = pd.read_csv(cleaned_path)

df['year'] = pd.to_numeric(df['year'], errors='coerce')
df_filtered = df[(df['year'] >= 1960) & (df['year'] <= 2022)]

filtered_path = os.path.join(output_dir, "merged_worldbank_long_1960_2022.csv")
df_filtered.to_csv(filtered_path, index=False)

In [ ]:
import pandas as pd
import os

output_dir = "output"
cleaned_path = os.path.join(output_dir, "merged_worldbank_long_cleaned.csv")
df = pd.read_csv(cleaned_path)

df['year'] = pd.to_numeric(df['year'], errors='coerce')
mask = (df['year'] >= 1960) & (df['year'] <= 2022)
subset = df[mask]

value_cols = subset.columns.difference([df.columns[0], 'year'])

nan_countries = subset.groupby(df.columns[0])[value_cols].apply(lambda x: x.isna().any(axis=1).any())

count_nan_countries = nan_countries.sum()

print("Количество стран с хотя бы одним NaN за 1960-2022:", count_nan_countries)


У России нет пропусков за 1960-2022.


In [35]:
import pandas as pd
import os

output_dir = "output"
filtered_path = os.path.join(output_dir, "merged_worldbank_long_1960_2022.csv")
df = pd.read_csv(filtered_path)

country_col = df.columns[0]
year_col = "year"

value_cols = df.columns.difference([country_col, year_col])

nan_rows = df[df[value_cols].isna().any(axis=1)]

for idx, row in nan_rows.iterrows():
    nan_cols = [col for col in value_cols if pd.isna(row[col])]
    print(f"{row[country_col]} | {row[year_col]} | {', '.join(nan_cols)}")


ABW | 1960 | gdppc
ABW | 1961 | gdppc
ABW | 1962 | gdppc
ABW | 1963 | gdppc
ABW | 1964 | gdppc
ABW | 1965 | gdppc
ABW | 1966 | gdppc
ABW | 1967 | gdppc
ABW | 1968 | gdppc
ABW | 1969 | gdppc
ABW | 1970 | gdppc
ABW | 1971 | gdppc
ABW | 1972 | gdppc
ABW | 1973 | gdppc
ABW | 1974 | gdppc
ABW | 1975 | gdppc
ABW | 1976 | gdppc
ABW | 1977 | gdppc
ABW | 1978 | gdppc
ABW | 1979 | gdppc
ABW | 1980 | gdppc
ABW | 1981 | gdppc
ABW | 1982 | gdppc
ABW | 1983 | gdppc
ABW | 1984 | gdppc
ABW | 1985 | gdppc
ABW | 1986 | gdppc
ABW | 1987 | gdppc
ABW | 1988 | gdppc
ABW | 1989 | gdppc
ABW | 1990 | gdppc
ABW | 1991 | gdppc
ABW | 1992 | gdppc
ABW | 1993 | gdppc
ABW | 1994 | gdppc
ABW | 1995 | gdppc
ABW | 1996 | gdppc
ABW | 1997 | gdppc
ABW | 1998 | gdppc
ABW | 1999 | gdppc
ABW | 2000 | gdppc
ABW | 2001 | gdppc
ABW | 2002 | gdppc
ABW | 2003 | gdppc
ABW | 2004 | gdppc
ABW | 2005 | gdppc
ABW | 2006 | gdppc
ABW | 2007 | gdppc
ABW | 2008 | gdppc
ABW | 2009 | gdppc
ABW | 2010 | gdppc
ABW | 2011 | gdppc
ABW | 2012 |

In [50]:
import pandas as pd
import os

output_dir = "output"
filtered_path = os.path.join(output_dir, "merged_worldbank_long_1960_2022.csv")
df = pd.read_csv(filtered_path)

country_col = df.columns[0]
year_col = "year"
value_cols = df.columns.difference([country_col, year_col])

nan_countries = df.groupby(country_col)[value_cols].apply(lambda x: x.isna().any().any())
good_countries = nan_countries[~nan_countries].index

df_clean = df[df[country_col].isin(good_countries)]

cleaned_path = os.path.join(output_dir, "merged_worldbank_long_no_nan.csv")
df_clean = df_clean.rename(columns={'economy': 'country_code'})
df_clean.to_csv(cleaned_path, index=False)
print("saved:", cleaned_path)
print(df_clean.columns)

saved: output/merged_worldbank_long_no_nan.csv
Index(['country_code', 'year', 'age_0_14_share', 'age_65_plus_share',
       'crude_birth_rate', 'crude_death_rate', 'fertility_rate',
       'life_expectancy', 'net_migration', 'population', 'urban_share',
       'gdppc'],
      dtype='object')


In [44]:
import pandas as pd
import os

output_dir = "output"
path = os.path.join(output_dir, "merged_worldbank_long_no_nan.csv")
df = pd.read_csv(path)

country_col = df.columns[0]
year_col = "year"

russia_rows = df[df[country_col].str.upper() == "RUS"]["gdppc"]

population_col = "population"
pop_2022 = df[df[year_col] == 2022]
good_countries = pop_2022[pop_2022[population_col] >= 1_000_000][country_col].unique()
df_filtered = df[df[country_col].isin(good_countries)]

df_filtered.to_csv(os.path.join(output_dir, "merged_worldbank_long_filtered.csv"), index=False)
country_col = df.columns[0]
num_countries = df[country_col].nunique()
print("количество стран:", num_countries)
print("ввп россии на душу с учётом инфляции\n", russia_rows)

количество стран: 144
ввп россии на душу с учётом инфляции
 7119     5557.000000
7120     5874.000000
7121     6229.000000
7122     6405.000000
7123     6754.000000
            ...     
7177    24690.082293
7178    25244.118678
7179    24625.384368
7180    26119.947639
7181    25437.108022
Name: gdppc, Length: 63, dtype: float64


In [ ]:
import pandas as pd

df = pd.read_csv("output/merged_worldbank_long_no_nan.csv")
russia_pop = df[(df['country_code'] == 'RUS') & (df['year'].between(2013, 2020))][['year', 'population']]
print("Russia 2013-2020:")
print(russia_pop)

# Украина
ukraine_pop = df[(df['country_code'] == 'UKR') & (df['year'].between(2013, 2020))][['year', 'population']]
print("\nUkraine 2013-2020:")
print(ukraine_pop)

Россия без Крыма(ща 2015 146,7 млн с крымом). Украина логично отсутсвует в датасете. Если без Крыма до 2014 учитывать а потом с Крымом данные некорректные будут